# Stokes equation

In this tutorial we present how to solve a three-fields Stokes equation with [PyGeoN](https://github.com/compgeo-mox/pygeon). The unknowns are the vorticity $r$, the velocity $q$ and the pressure $p$.

Let $\Omega=(0,1)^2$ with boundary $\partial \Omega$ and outward unit normal ${\nu}$. Given 
$\mu$ the viscosity, we want to solve the following problem: find $(r, q, p)$ such that
\begin{align*}
&\mu^{-1} r - \nabla \times q = 0 \\
&\nabla \times r + \nabla p = {0} \\
&\nabla \cdot {q} = 0
\end{align*}
with boundary conditions:
$$
\nu \cdot q = 0 \text{ on } \partial \Omega \qquad
\nu \times q = 1 \text{ on } \partial_{top} \Omega \qquad
\nu \times q = 0 \text{ on } \partial \Omega \setminus \partial_{top} \Omega
$$
that is a lid-driven cavity. The normal velocity is an essential condition on the whole boundary, while the tangential velocity $\nu \times q$ is a natural condition, imposed through the first equation: the lid moves with unit velocity in the negative $x$ direction. Since only the velocity is prescribed on the boundary, the pressure is defined up to a constant: we fix its value in a cell, and other choices can be made.

We present *step-by-step* how to create the grid, declare the problem data, and finally solve the problem.

First we import some of the standard modules. Since PyGeoN is based on [PorePy](https://github.com/pmgbergen/porepy) we import both modules.

In [ ]:
import os

import numpy as np
import porepy as pp
import scipy.sparse as sps

import pygeon as pg

We create now the grid, since we will use linear Lagrange elements for $r$ and Raviart-Thomas for ${q}$ we are restricted to triangles. In this example we consider a bi-dimensional grid, but the presented code will work also in 3d where $r$ is approximated with Nédélec elements.

In [ ]:
mesh_size = 0.1
dim = 2

sd = pg.unit_grid(dim, mesh_size, as_mdg=False)
sd.compute_geometry()

We define now the finite element spaces for all the variables.

In [ ]:
# finite element spaces
rt0 = pg.RT0("velocity")
l1 = pg.Lagrange1("vorticity")  # pg.NedelecR0
p0 = pg.PwConstants("pressure")

# build the degrees of freedom
dofs = np.array([l1.ndof(sd), rt0.ndof(sd), p0.ndof(sd)])

With the following code we set the data, in particular the viscosity and the boundary conditions. Since we need to identify each side of $\partial \Omega$ for $r$ and $q$ we need few steps.

In [ ]:
# with the following steps we identify the portions of the boundary
# to impose the boundary conditions
ess_faces = sd.tags["domain_boundary_faces"]
top = np.isclose(sd.face_centers[1, :], 1)

ess_ridges = np.zeros(sd.num_ridges, dtype=bool)

ess_cells = np.zeros(sd.num_cells, dtype=bool)
ess_cells[0] = True

bc_faces = np.zeros(sd.num_faces)
# the tangential velocity on the top, a natural condition for the vorticity
bc_ridges = l1.assemble_nat_bc(sd, lambda _: 1, top)

bc_val = np.hstack((bc_ridges, bc_faces, np.zeros(sd.num_cells)))
bc_ess = np.hstack((ess_ridges, ess_faces, ess_cells))

parameters = {
    pg.SECOND_ORDER_TENSOR: pp.SecondOrderTensor(np.ones(sd.num_cells)),
}
data = pp.initialize_data({}, "vorticity", parameters)

Once the data are assigned, we construct the matrices. In particular, the linear system associated with the equation is given as
$$
\left(
\begin{array}{cc} 
M & -B^\top_r & \\
B_r & & -B^\top_q\\
& B_q &
\end{array}
\right)
\left(
\begin{array}{c} 
r\\
q\\ 
p
\end{array}
\right)
=\left(
\begin{array}{c} 
q_{\partial}\\
p_{\partial}\\ 
0
\end{array}
\right)
$$<br>
where $q_{\partial}$ and $p_{\partial}$ are the vectors associated with the natural boundary conditions on the tangential velocity and on the pressure, respectively; here $p_{\partial} = 0$, since the normal velocity is prescribed on the whole boundary. To construct the saddle-point problem, we rely on the `scipy.sparse` function `block_array`. Once the matrix is created, we also construct the right-hand side containing the boundary conditions.

In [ ]:
# construct the local matrices
mass = l1.assemble_mass_matrix(sd, data)
face_mass = rt0.assemble_mass_matrix(sd)
cell_mass = p0.assemble_mass_matrix(sd)

curl = face_mass @ l1.assemble_diff_matrix(sd)
div = cell_mass @ rt0.assemble_diff_matrix(sd)

# assemble the saddle point problem
spp = sps.block_array(
    [
        [mass, -curl.T, None],
        [curl, None, -div.T],
        [None, div, None],
    ],
).tocsc()

# assemble the right-hand side
rhs = bc_val

We solve the linear system and extract the three solutions $r$, $q$ and $p$.

In [ ]:
# solve the problem
ls = pg.LinearSystem(spp, rhs)
ls.flag_ess_bc(bc_ess, np.zeros(bc_ess.size))
x = ls.solve()

# split the solution into the components
idx = np.cumsum(dofs[:-1])
r, q, p = np.split(x, idx)

Since the computed $q$ is one value per facet of the grid, for visualization purposes we project the velocity, as well as the pressure, in each cell, while the vorticity $r$, one value per node, is exported at the nodes. We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [ ]:
# post process velocity
proj_q = rt0.eval_at_cell_centers(sd)
cell_q = (proj_q @ q).reshape((3, -1))

# post process pressure
proj_p = p0.eval_at_cell_centers(sd)
cell_p = proj_p @ p

# export the solution
folder_name = os.path.join(os.getcwd(), "stokes")
file_name = "sol"

save = pp.Exporter(sd, file_name, folder_name=folder_name)
save.write_vtu([("cell_q", cell_q), ("cell_p", cell_p)], data_pt=[("r", r)])

A representation of the computed solution is given below.

<div style="text-align: center;">
  <img src="../fig/stokes.png" alt="Stokes solution"/>
</div>

In [ ]:
# Consistency check
assert np.isclose(np.linalg.norm(r), 65.20686352458148)
assert np.isclose(np.linalg.norm(q), 0.3334007092723473)
assert np.isclose(np.linalg.norm(p), 2.1672295538126964)